# Descartes-MMRec: Methodic Doubt-Aware Multimodal Recommendation

Dự án này triển khai thuật toán gợi ý đa phương thức lấy cảm hứng từ triết lý "Sự hoài nghi có phương pháp" của René Descartes.
Mô hình thực hiện 3 giai đoạn:
1. **Doubt Pruning:** Cắt bỏ tương tác nhiễu (Doubt Evaluator).
2. **Adversarial Discovery:** Tạo nhiễu cường hóa đặc trưng (FGSM).
3. **Robust Generative Diffusion:** Khử nhiễu dựa trên Core Graph.

**Lưu ý:** Bật T4 GPU trước khi chạy (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
# 1. Kết nối Google Drive để lưu/tải Checkpoint và Dataset
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone mã nguồn từ Github
import os
GITHUB_REPO = "https://github.com/thyelmot/Descartes_MMRec_Colab.git"
WORK_DIR = "/content/Descartes_MMRec_Colab"

if os.path.exists(WORK_DIR):
    print('Folder đã tồn tại, đang pull bản mới nhất...')
    !cd {WORK_DIR} && git pull
else:
    !git clone {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}

# 3. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt

In [ ]:
# 4. Cấu hình Đường dẫn Dataset và Checkpoint
import yaml

# Đường dẫn thư mục Datasets trên Google Drive (đổi lại nếu bạn để ở nơi khác)
DATASET_DRIVE_PATH = "/content/drive/MyDrive/DiffMM_Data"

# Thư mục lưu kết quả mô hình
CHECKPOINT_DIR = "/content/drive/MyDrive/DiffMM_Checkpoints"
!mkdir -p {CHECKPOINT_DIR}

In [ ]:
# 5. Cấu hình Hyperparameter và Huấn luyện (Giai đoạn 1+2+3)
DATASET = "baby"  # Có thể chọn: baby, sports, tiktok

CONFIG = {
    "dataset": DATASET,
    "epoch": 50,           # Tổng số vòng lặp
    "batch": 1024,         # Batch size (giữ nguyên như DiffMM baseline)
    "tstEpoch": 1,         # Cứ 1 vòng lặp thì đánh giá (Test) 1 lần
    
    # --- DiffMM Baseline Hyperparameters (Baby) ---
    "keepRate": 1.0,       # baby: 1.0 | sports: 0.5 | tiktok: 0.5
    "ssl_reg": 0.1,        # baby: 1e-1 | sports: 1e-2 | tiktok: 1e-2
    "e_loss": 0.01,        # baby: 0.01 | sports: 0.01 | tiktok: 0.1
    
    # --- Descartes-MMRec Hyperparameters ---
    "omega": 0.5,          # Hệ số khuếch đại nhiễu (nhẹ nhàng hơn V1)
    "alpha_doubt": 0.3,    # Trọng số bất đồng thuận nội tại (Visual vs Text)
    "beta_doubt": 0.5,     # Trọng số lệch pha Sở thích User
    "gamma_doubt": 0.2,    # Trọng số Popularity Bias
    "epsilon_adv": 0.01,   # Sức mạnh tấn công FGSM (nhẹ)
    "lambda_adv": 0.03,    # Trọng số Adversarial Loss (nhẹ)
    "adv_warmup_epochs": 10,# Chờ 10 epochs trước khi bật adversarial
    # ---------------------------------------
}

# Cập nhật argument chạy code
args_str = f"--data {CONFIG["dataset"]} --epoch {CONFIG["epoch"]} --batch {CONFIG["batch"]} --tstEpoch {CONFIG["tstEpoch"]} "
args_str += f"--keepRate {CONFIG["keepRate"]} --ssl_reg {CONFIG["ssl_reg"]} --e_loss {CONFIG["e_loss"]} "
args_str += f"--omega {CONFIG["omega"]} --alpha_doubt {CONFIG["alpha_doubt"]} "
args_str += f"--beta_doubt {CONFIG["beta_doubt"]} --gamma_doubt {CONFIG["gamma_doubt"]} "
args_str += f"--epsilon_adv {CONFIG["epsilon_adv"]} --lambda_adv {CONFIG["lambda_adv"]} "
args_str += f"--adv_warmup_epochs {CONFIG["adv_warmup_epochs"]} "
args_str += f"--dataset_path {DATASET_DRIVE_PATH} --model_type descartes_mmrec --checkpoint_dir {CHECKPOINT_DIR}"

print(f"Đang huấn luyện mô hình Descartes-MMRec với lệnh:\npython Main.py {args_str}\n")
!python Main.py {args_str}

In [ ]:
# 6. (Nâng cao) Phân tích Core Graph Sparsity và Loss Curve
# Nếu bạn muốn theo dõi chi tiết hiệu suất trong quá trình huấn luyện, chạy đoạn code sau.
import matplotlib.pyplot as plt
import re
import os

log_file = os.path.join(CHECKPOINT_DIR, f"{DATASET}_descartes.log")

if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        content = f.read()
        
    epochs = [int(x) for x in re.findall(r'Epoch (\d+)', content) if int(x) > 0]
    recalls = [float(x) for x in re.findall(r'Recall : (\d+\.\d+)', content)]
    ndcgs = [float(x) for x in re.findall(r'NDCG : (\d+\.\d+)', content)]
    
    min_len = min(len(epochs), len(recalls))
    if min_len > 0:
        plt.figure(figsize=(10, 4))
        plt.plot(epochs[:min_len], recalls[:min_len], marker='o', label='Recall@20', color='blue')
        plt.plot(epochs[:min_len], ndcgs[:min_len], marker='s', label='NDCG@20', color='orange')
        plt.title('Performance Metrics over Epochs')
        plt.xlabel('Epoch')
        plt.ylabel('Score')
        plt.legend()
        plt.grid(True)
        plt.show()
else:
    print("Chưa tìm thấy log file. Vui lòng chạy Cell 4 trước.")